# BailMitra — End-to-End Risk Assessment Pipeline

This notebook is the "assembly" pipeline for BailMitra. It:

1. Loads the **ILDC-style** case-text corpus (`ildc_dataset.csv`) and extracts simple
   textual signals (prior convictions, bail history, charge severity) with regex, then
   trains a Random Forest to turn those signals into a `risk_score`.
2. Loads the **PredEx-style** dataset (`predex_dataset.csv`) and fits a Linear Regression
   between `risk_score` and `case_outcome` to learn a global outcome weight.
3. Combines both signals into a `weighted_score` and a final `Low / Medium / High`
   `final_risk_level` for every ILDC case, saved to `final_risk_assessment.csv`.
4. Loads the tabular **structured-features** production model (`risk_prediction_model.pkl`,
   trained in `risk_assess.ipynb` on `dummy_criminal_risk_dataset.csv`) and exposes a single
   `assess_bail_risk(...)` function that a form-based BailMitra front-end can call directly.

**Note on data provenance:** `ildc_dataset.csv` and `predex_dataset.csv` shipped alongside
this notebook are lightweight **synthetic stand-ins** for the real ILDC (Indian Legal
Documents Corpus) and PredEx datasets, generated so the whole pipeline runs offline and
end-to-end. Swap in the real datasets (e.g. via `datasets.load_dataset("L-NLProc/PredEx")`)
for production/legal-research use — the cells below are already written the same way the
original notebook loaded them, so only the CSV paths need to change.

In [1]:
import re
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", 25)
RANDOM_STATE = 42

## 1. Load the ILDC-style case corpus

In [ ]:
ildc_df = pd.read_csv("input/ildc_dataset.csv")
print(ildc_df.shape)
ildc_df.head()

(12000, 5)


,case_id,court,year,case_text,risk_category
0,ILDC200000,Sessions Court,2020,"In the matter concerning the petitioner, charg...",Low
1,ILDC200001,High Court,2016,"In the matter concerning the petitioner, charg...",Medium
2,ILDC200002,District Court,2015,"In the matter concerning the respondent, charg...",Low
3,ILDC200003,High Court,2008,"In the matter concerning the defendant, charge...",Medium
4,ILDC200004,Supreme Court,2010,"In the matter concerning the appellant, charge...",High


## 2. Regex-based feature extraction from case text

Lightweight NLP signal extraction — the same approach as the original notebook, counting
mentions of prior convictions, bail history, and charge severity within each case's text.

In [3]:
def extract_features(text):
    prior_convictions = len(re.findall(r"prior conviction|previous offense", text, re.IGNORECASE))
    bail_history = len(re.findall(r"bail granted|bail denied", text, re.IGNORECASE))
    charge_severity = len(re.findall(r"serious charge|minor charge", text, re.IGNORECASE))
    return pd.Series([prior_convictions, bail_history, charge_severity])

ildc_df[["prior_convictions", "bail_history", "charge_severity"]] = (
    ildc_df["case_text"].apply(extract_features)
)
ildc_df.head()

,case_id,court,year,case_text,risk_category,prior_convictions,bail_history,charge_severity
0,ILDC200000,Sessions Court,2020,"In the matter concerning the petitioner, charg...",Low,0,0,0
1,ILDC200001,High Court,2016,"In the matter concerning the petitioner, charg...",Medium,1,1,1
2,ILDC200002,District Court,2015,"In the matter concerning the respondent, charg...",Low,0,0,0
3,ILDC200003,High Court,2008,"In the matter concerning the defendant, charge...",Medium,1,1,0
4,ILDC200004,Supreme Court,2010,"In the matter concerning the appellant, charge...",High,1,1,1


## 3. Encode labels and train a Random Forest to assign a `risk_score`

In [4]:
risk_order = {"Low": 0, "Medium": 1, "High": 2}
ildc_df["risk_category_encoded"] = ildc_df["risk_category"].map(risk_order)
print(risk_order)

X = ildc_df[["prior_convictions", "bail_history", "charge_severity"]]
y = ildc_df["risk_category_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

rf_text_model = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=RANDOM_STATE)
rf_text_model.fit(X_train, y_train)

ildc_df["risk_score"] = rf_text_model.predict(X)
print(f"Train accuracy: {rf_text_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {rf_text_model.score(X_test, y_test):.3f}")

{'Low': 0, 'Medium': 1, 'High': 2}


Train accuracy: 0.918
Test accuracy:  0.926


## 4. Load the PredEx-style dataset and learn an outcome weight

In [ ]:
predex_df = pd.read_csv("input/predex_dataset.csv")
print(predex_df.shape)
predex_df.head()

(50000, 4)


,case_id,risk_score,explanation_length,case_outcome
0,PDX300000,2,367,1
1,PDX300001,1,119,0
2,PDX300002,1,145,0
3,PDX300003,1,148,1
4,PDX300004,1,445,1


In [6]:
X_predex = predex_df[["risk_score"]]
y_predex = predex_df["case_outcome"]

lr_model = LinearRegression()
lr_model.fit(X_predex, y_predex)

predex_df["weight"] = lr_model.coef_[0]
print(f"Learned risk_score -> case_outcome coefficient: {lr_model.coef_[0]:.4f}")

Learned risk_score -> case_outcome coefficient: 0.3469


## 5. Combine both signals into a weighted, final risk level

`risk_score` only takes the discrete values 0 / 1 / 2 (Low / Medium / High), so
`weighted_score` inherits that discreteness. Thresholds are set as fractions of the
maximum observed score (1/3 and 2/3) rather than fixed absolute cutoffs, so the mapping
stays correct even if the learned `weight` scale changes.

In [7]:
ildc_df["weighted_score"] = ildc_df["risk_score"] * predex_df["weight"].mean()

max_score = ildc_df["weighted_score"].max()
low_cut, high_cut = max_score / 3, 2 * max_score / 3
print(f"Low/Medium cutoff: {low_cut:.4f}  |  Medium/High cutoff: {high_cut:.4f}")

def assign_final_risk(score):
    if score >= high_cut:
        return "High"
    elif score >= low_cut:
        return "Medium"
    else:
        return "Low"

ildc_df["final_risk_level"] = ildc_df["weighted_score"].apply(assign_final_risk)
ildc_df["final_risk_level"].value_counts()

Low/Medium cutoff: 0.2313  |  Medium/High cutoff: 0.4625


final_risk_level
Low       6525
Medium    2827
High      2648
Name: count, dtype: int64

In [8]:
ildc_df.to_csv("final_risk_assessment.csv", index=False)
print("Risk assessment completed and saved to final_risk_assessment.csv")
ildc_df[["case_id", "risk_category", "risk_score", "weighted_score", "final_risk_level"]].head(10)

Risk assessment completed and saved to final_risk_assessment.csv


,case_id,risk_category,risk_score,weighted_score,final_risk_level
0,ILDC200000,Low,0,0.000000,Low
1,ILDC200001,Medium,2,0.693787,High
2,ILDC200002,Low,0,0.000000,Low
3,ILDC200003,Medium,1,0.346893,Medium
4,ILDC200004,High,2,0.693787,High
5,ILDC200005,Medium,1,0.346893,Medium
6,ILDC200006,Medium,2,0.693787,High
7,ILDC200007,Low,0,0.000000,Low
8,ILDC200008,Low,0,0.000000,Low
9,ILDC200009,Low,0,0.000000,Low


## 6. Structured-feature production model

Load the Random Forest trained in `risk_assess.ipynb` on the 20-feature
`dummy_criminal_risk_dataset.csv`, and its label encoders, so a BailMitra front-end
(web form / API) can score a new case directly from structured fields instead of
free-text.

In [ ]:
risk_model = joblib.load("input/risk_prediction_model.pkl")
feature_label_encoders = joblib.load("feature_label_encoders.pkl")
risk_target_encoder = joblib.load("risk_target_encoder.pkl")

FEATURE_COLS = [
    "Age", "Gender", "Prior_Arrests", "Prior_Convictions", "Offense_Severity",
    "Charge_Type", "Bail_History", "Flight_Risk", "Employment_Status",
    "Community_Ties", "Weapon_Used", "Violence_Involved", "Evidence_Strength",
    "Victim_Impact", "Repeat_Offender", "Substance_Abuse_History",
    "Mental_Health_Concern", "Court_Appearance_History",
]

print("Model and encoders loaded OK.")

Model and encoders loaded OK.


## 7. `assess_bail_risk(...)` — single entry point for the BailMitra UI/API

In [10]:
def assess_bail_risk(age, gender, prior_arrests, prior_convictions, offense_severity,
                      charge_type, bail_history, flight_risk, employment_status,
                      community_ties, weapon_used, violence_involved, evidence_strength,
                      victim_impact, repeat_offender, substance_abuse_history,
                      mental_health_concern, court_appearance_history):
    """Score a single case against the structured-feature production model and
    return a dict with the predicted risk category and class probabilities."""
    raw = {
        "Age": age, "Gender": gender, "Prior_Arrests": prior_arrests,
        "Prior_Convictions": prior_convictions, "Offense_Severity": offense_severity,
        "Charge_Type": charge_type, "Bail_History": bail_history, "Flight_Risk": flight_risk,
        "Employment_Status": employment_status, "Community_Ties": community_ties,
        "Weapon_Used": weapon_used, "Violence_Involved": violence_involved,
        "Evidence_Strength": evidence_strength, "Victim_Impact": victim_impact,
        "Repeat_Offender": repeat_offender, "Substance_Abuse_History": substance_abuse_history,
        "Mental_Health_Concern": mental_health_concern,
        "Court_Appearance_History": court_appearance_history,
    }

    row = {}
    for key, value in raw.items():
        if key in feature_label_encoders:
            row[key] = feature_label_encoders[key].transform([value])[0]
        else:
            row[key] = value

    input_df = pd.DataFrame([row])[FEATURE_COLS]
    pred_encoded = risk_model.predict(input_df)[0]
    probabilities = risk_model.predict_proba(input_df)[0]

    return {
        "risk_category": risk_target_encoder.inverse_transform([pred_encoded])[0],
        "probabilities": dict(zip(risk_target_encoder.classes_, probabilities.round(3))),
    }


# Example: a moderate-profile case
result = assess_bail_risk(
    age=29, gender="Male", prior_arrests=2, prior_convictions=1,
    offense_severity="Moderate", charge_type="Misdemeanor", bail_history="No Prior Bail",
    flight_risk="Medium", employment_status="Self-Employed", community_ties="Moderate",
    weapon_used="No", violence_involved="No", evidence_strength="Moderate",
    victim_impact="Minor", repeat_offender="No", substance_abuse_history="No",
    mental_health_concern="No", court_appearance_history="Always Appeared",
)
result

{'risk_category': 'Low',
 'probabilities': {'High': np.float64(0.009),
  'Low': np.float64(0.896),
  'Medium': np.float64(0.095)}}

## 8. Batch-score a few example profiles

In [11]:
example_profiles = [
    dict(age=45, gender="Female", prior_arrests=0, prior_convictions=0,
         offense_severity="Minor", charge_type="Petty Offense", bail_history="No Prior Bail",
         flight_risk="Low", employment_status="Employed", community_ties="Strong",
         weapon_used="No", violence_involved="No", evidence_strength="Weak",
         victim_impact="No Impact", repeat_offender="No", substance_abuse_history="No",
         mental_health_concern="No", court_appearance_history="Always Appeared"),
    dict(age=27, gender="Male", prior_arrests=8, prior_convictions=5,
         offense_severity="Severe", charge_type="Serious Felony", bail_history="Bail Violated",
         flight_risk="High", employment_status="Unemployed", community_ties="Weak",
         weapon_used="Yes", violence_involved="Yes", evidence_strength="Strong",
         victim_impact="Severe", repeat_offender="Yes", substance_abuse_history="Yes",
         mental_health_concern="Yes", court_appearance_history="Frequently Missed"),
]

for i, profile in enumerate(example_profiles, start=1):
    res = assess_bail_risk(**profile)
    print(f"Profile {i}: {res['risk_category']}  |  probabilities={res['probabilities']}")

Profile 1: Low  |  probabilities={'High': np.float64(0.0), 'Low': np.float64(0.996), 'Medium': np.float64(0.004)}
Profile 2: High  |  probabilities={'High': np.float64(0.946), 'Low': np.float64(0.006), 'Medium': np.float64(0.048)}


## 9. (Optional) Real LLM-based qualitative second opinion

The original notebook explored calling an external LLM (Groq / Llama-3.3) to produce a
free-text risk opinion from a natural-language case description, as a qualitative
complement to the structured model above. That integration needs a personal API key
and network access, so it's left here as a template rather than executed automatically.

**Security note:** never hard-code API keys directly in a notebook — load them from an
environment variable or a local `.env` file that is excluded from version control.

In [12]:
# import os
# from groq import Groq
#
# client = Groq(api_key=os.environ["GROQ_API_KEY"])  # set this in your shell/.env, never hard-code it
#
# def qualitative_risk_opinion(case_description: str) -> str:
#     chat_completion = client.chat.completions.create(
#         messages=[{
#             "role": "user",
#             "content": (
#                 f"{case_description} Based on the provided details, assess the risk level "
#                 "for granting bail. Only return one of the following labels: "
#                 "'Low Risk', 'Moderate Risk', or 'High Risk'. Do not provide any explanation."
#             ),
#         }],
#         model="llama-3.3-70b-versatile",
#     )
#     return chat_completion.choices[0].message.content
#
# print(qualitative_risk_opinion("John Doe, a 35-year-old male, was convicted of armed robbery..."))